# Feature Engineering
The process of creating, transforming, or selecting input variables (features) from raw data to improve the performance of machine learning models. It often involves techniques like scaling, encoding categorical data, creating new features from existing ones, or reducing dimensionality. Essentially, it’s about turning raw data into meaningful inputs that help models learn patterns more effectively.

Create meaningful features:
- One-hot encoding
- Extract date features (weekday, month, hour…)
- Domain logic features
- Combine or derive features (ratios, interactions)
- Transform skew (log transform)
- Text preprocessing (tokenisation, embeddings)

Here we will be using pandas to handle storing and processing the data

In [48]:
import pandas as pd

df = pd.read_csv("data/nba_games.csv",index_col=0)

In [50]:
df["won_opp"] = ~df["won"]

In [51]:
df.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,usg%_max_opp,ortg_max_opp,drtg_max_opp,team_opp,total_opp,home_opp,season,date,won,won_opp
2630,240.0,39.0,75.0,0.520,13.0,19.0,0.684,22.0,22.0,1.000,...,29.7,150.0,133.0,UTA,104,0,2017,2016-10-25,True,False
2631,240.0,40.0,82.0,0.488,8.0,24.0,0.333,16.0,16.0,1.000,...,39.2,220.0,120.0,POR,113,1,2017,2016-10-25,False,True
2632,240.0,40.0,85.0,0.471,7.0,33.0,0.212,13.0,18.0,0.722,...,36.2,263.0,112.0,SAS,129,0,2017,2016-10-25,False,True
2633,240.0,47.0,98.0,0.480,12.0,24.0,0.500,23.0,26.0,0.885,...,31.8,139.0,143.0,GSW,100,1,2017,2016-10-25,True,False
2634,240.0,45.0,94.0,0.479,13.0,35.0,0.371,14.0,19.0,0.737,...,36.8,153.0,125.0,NYK,88,0,2017,2016-10-25,True,False


## Rolling averages
First useful feature to add to our dataframe is the rolling averages which our model can use to see how each team is performing over their last 10 games

In [53]:
removed_columns = ["season", "date", "team", "team_opp"]
selected_columns = list(df.columns[~df.columns.isin(removed_columns)])
opp_columns = [c for c in selected_columns if "_opp" in c]
team_columns = df.columns[~df.columns.isin(removed_columns + opp_columns)]

In [54]:
def find_team_averages(group,n):
    # Compute rolling mean with an n-game window, shifted so current row isn’t included
    return group.shift().rolling(n, min_periods=1).mean()

def get_rolling_df(df,keys,cols_to_roll,n):
    df_rolling = find_team_averages(df.groupby(keys, group_keys=False)[cols_to_roll],n)
    rolling_cols = [f"{col}_{n}" for col in df_rolling.columns]
    df_rolling.columns = rolling_cols
    return df_rolling

df_rolling_10 = get_rolling_df(df,["team","season"],team_columns,10)
df_rolling_5 = get_rolling_df(df,["team","season"],team_columns,5)

df_rolling_10_opp = get_rolling_df(df,["team_opp","season"],opp_columns,10)
df_rolling_5_opp = get_rolling_df(df,["team_opp","season"],opp_columns,5)

df = pd.concat([df, df_rolling_5, df_rolling_10, df_rolling_10_opp, df_rolling_5_opp], axis=1)

In [59]:
df.tail()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,ast%_max_opp_5,stl%_max_opp_5,blk%_max_opp_5,tov%_max_opp_5,usg%_max_opp_5,ortg_max_opp_5,drtg_max_opp_5,total_opp_5,home_opp_5,won_opp_5
10473,240.0,35.0,78.0,0.449,8.0,27.0,0.296,14.0,21.0,0.667,...,36.50,5.80,12.58,43.92,39.74,183.0,123.2,110.8,0.4,0.4
10474,240.0,38.0,82.0,0.463,20.0,42.0,0.476,10.0,14.0,0.714,...,35.00,4.28,13.08,41.12,39.54,183.0,118.2,110.0,0.4,0.6
10475,240.0,38.0,85.0,0.447,8.0,32.0,0.250,21.0,27.0,0.778,...,37.14,4.38,14.24,41.12,40.56,172.4,118.6,106.6,0.6,0.4
10476,240.0,39.0,82.0,0.476,13.0,33.0,0.394,23.0,29.0,0.793,...,39.32,3.86,15.98,45.12,40.66,187.8,119.8,107.0,0.4,0.6
10477,240.0,39.0,80.0,0.488,11.0,31.0,0.355,21.0,30.0,0.700,...,36.76,3.96,12.74,42.98,32.44,179.8,117.8,106.2,0.4,0.6


In [60]:
df = df.dropna() #Drop all null/na values

In [62]:
df.describe()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,ast%_max_opp_5,stl%_max_opp_5,blk%_max_opp_5,tov%_max_opp_5,usg%_max_opp_5,ortg_max_opp_5,drtg_max_opp_5,total_opp_5,home_opp_5,won_opp_5
count,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,...,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000,7778.000000
mean,241.555670,39.816791,86.754693,0.459893,10.530985,29.444973,0.357307,17.430188,22.668038,0.769279,...,42.570469,5.861133,8.753269,39.618747,34.748933,176.726541,117.980068,107.663772,0.500084,0.504065
std,7.204613,5.175211,7.308693,0.054130,3.656335,7.180752,0.090388,6.012659,7.290716,0.102539,...,6.619210,1.821637,2.948607,9.160975,3.851200,18.197798,5.003349,6.110255,0.207053,0.209505
min,240.000000,21.000000,61.000000,0.278000,1.000000,7.000000,0.053000,1.000000,1.000000,0.143000,...,23.500000,2.100000,2.200000,18.200000,26.400000,123.600000,98.000000,86.400000,0.000000,0.000000
25%,240.000000,36.000000,82.000000,0.422000,8.000000,24.000000,0.296000,13.000000,18.000000,0.706000,...,37.880000,4.760000,6.845000,32.760000,32.260000,163.600000,114.600000,103.600000,0.400000,0.400000
50%,240.000000,40.000000,87.000000,0.459000,10.000000,29.000000,0.356000,17.000000,22.000000,0.778000,...,41.830000,5.500000,8.180000,38.320000,33.960000,175.200000,118.000000,107.400000,0.450000,0.600000
75%,240.000000,43.000000,91.000000,0.494000,13.000000,34.000000,0.417000,21.000000,27.000000,0.840000,...,46.500000,6.460000,9.900000,45.220000,36.335000,188.200000,121.400000,111.800000,0.600000,0.600000
max,340.000000,61.000000,128.000000,0.667000,27.000000,70.000000,0.842000,44.000000,64.000000,1.000000,...,69.760000,25.260000,29.620000,100.000000,66.240000,274.600000,140.000000,138.000000,1.000000,1.000000


In [63]:
for col in df.columns:
    print(col)

mp
fg
fga
fg%
3p
3pa
3p%
ft
fta
ft%
orb
drb
trb
ast
stl
blk
tov
pf
pts
ts%
efg%
3par
ftr
orb%
drb%
trb%
ast%
stl%
blk%
tov%
usg%
ortg
drtg
fg_max
fga_max
fg%_max
3p_max
3pa_max
3p%_max
ft_max
fta_max
ft%_max
orb_max
drb_max
trb_max
ast_max
stl_max
blk_max
tov_max
pf_max
pts_max
gmsc_max
+/-_max
ts%_max
efg%_max
3par_max
ftr_max
orb%_max
drb%_max
trb%_max
ast%_max
stl%_max
blk%_max
tov%_max
usg%_max
ortg_max
drtg_max
team
total
home
mp_opp
fg_opp
fga_opp
fg%_opp
3p_opp
3pa_opp
3p%_opp
ft_opp
fta_opp
ft%_opp
orb_opp
drb_opp
trb_opp
ast_opp
stl_opp
blk_opp
tov_opp
pf_opp
pts_opp
ts%_opp
efg%_opp
3par_opp
ftr_opp
orb%_opp
drb%_opp
trb%_opp
ast%_opp
stl%_opp
blk%_opp
tov%_opp
usg%_opp
ortg_opp
drtg_opp
fg_max_opp
fga_max_opp
fg%_max_opp
3p_max_opp
3pa_max_opp
3p%_max_opp
ft_max_opp
fta_max_opp
ft%_max_opp
orb_max_opp
drb_max_opp
trb_max_opp
ast_max_opp
stl_max_opp
blk_max_opp
tov_max_opp
pf_max_opp
pts_max_opp
gmsc_max_opp
+/-_max_opp
ts%_max_opp
efg%_max_opp
3par_max_opp
ftr_max_opp
orb%_m

In [65]:
df[["team","team_opp","won_10","won_opp_10","date"]]

,team,team_opp,won_10,won_opp_10,date
2656,SAC,SAS,1.0,1.0,2016-10-27
2657,SAS,SAC,1.0,1.0,2016-10-27
2658,LAC,POR,1.0,1.0,2016-10-27
2659,POR,LAC,1.0,1.0,2016-10-27
2660,BOS,CHI,1.0,1.0,2016-10-27
...,...,...,...,...,...
10473,GSW,TOR,0.6,0.6,2019-06-07
10474,GSW,TOR,0.5,0.7,2019-06-10
10475,TOR,GSW,0.6,0.6,2019-06-10
10476,TOR,GSW,0.5,0.6,2019-06-13


### Adding days rested since last game

In [92]:
def set_days_rested(df,team,prev_date,days_rested):
    # Sort within each team by date
    df = df.sort_values([team, "date"])

    # Compute previous game date for each team
    df[prev_date] = df.groupby(team)["date"].shift(1)
    
    #Set the dates as datetime
    df["date"] = pd.to_datetime(df["date"])
    df[prev_date] = pd.to_datetime(df[prev_date])

    # Difference in days
    df[days_rested] = (df["date"] - df[prev_date]).dt.days

    # First game of the season has no previous game → fill with something reasonable
    df[days_rested] = df[days_rested].fillna(0)

    return df

def set_both_teams_days_rested(df):
    df = set_days_rested(df,"team","prev_date","days_rested")
    df = set_days_rested(df,"team_opp","prev_date_opp","days_rested_opp")
    return df

df = set_both_teams_days_rested(df)

In [93]:
df = df.dropna()
df = df.sort_values("date")

In [94]:
df = df[df["days_rested"] <= 4]
df = df[df["days_rested_opp"] <= 4]

In [95]:
df[["team","team_opp","home","season","days_rested","days_rested_opp","won"]].head()

,team,team_opp,home,season,days_rested,days_rested_opp,won
2696,IND,CHI,0,2017,1.0,2.0,False
2681,NOP,SAS,0,2017,1.0,2.0,False
2694,BOS,CHO,0,2017,2.0,1.0,True
2692,ORL,CLE,0,2017,1.0,1.0,False
2688,SAS,NOP,1,2017,2.0,1.0,True


In [96]:
df.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,usg%_max_opp_10,ortg_max_opp_10,drtg_max_opp_10,total_opp_10,home_opp_10,won_10,prev_date,days_rested,prev_date_opp,days_rested_opp
2696,240.0,43.0,88.0,0.489,7.0,18.0,0.389,8.0,9.0,0.889,...,33.75,189.5,111.0,112.0,0.5,0.5,2016-10-28,1.0,2016-10-27,2.0
2681,240.0,32.0,86.0,0.372,4.0,22.0,0.182,11.0,15.0,0.733,...,27.50,225.0,107.5,114.5,0.0,0.0,2016-10-28,1.0,2016-10-27,2.0
2694,240.0,39.0,81.0,0.481,15.0,31.0,0.484,11.0,17.0,0.647,...,30.65,196.5,117.5,111.0,0.5,0.5,2016-10-27,2.0,2016-10-28,1.0
2692,240.0,38.0,84.0,0.452,8.0,28.0,0.286,15.0,26.0,0.577,...,31.65,188.0,105.0,108.0,0.5,0.0,2016-10-28,1.0,2016-10-28,1.0
2688,240.0,35.0,83.0,0.422,10.0,24.0,0.417,18.0,23.0,0.783,...,36.20,131.5,131.5,97.0,1.0,1.0,2016-10-27,2.0,2016-10-28,1.0


In [97]:
df[df["season"] == 2017].count()

mp                 2491
fg                 2491
fga                2491
fg%                2491
3p                 2491
                   ... 
won_10             2491
prev_date          2491
days_rested        2491
prev_date_opp      2491
days_rested_opp    2491
Length: 286, dtype: int64

In [98]:
for col in df.columns:
    print(col)

mp
fg
fga
fg%
3p
3pa
3p%
ft
fta
ft%
orb
drb
trb
ast
stl
blk
tov
pf
pts
ts%
efg%
3par
ftr
orb%
drb%
trb%
ast%
stl%
blk%
tov%
usg%
ortg
drtg
fg_max
fga_max
fg%_max
3p_max
3pa_max
3p%_max
ft_max
fta_max
ft%_max
orb_max
drb_max
trb_max
ast_max
stl_max
blk_max
tov_max
pf_max
pts_max
gmsc_max
+/-_max
ts%_max
efg%_max
3par_max
ftr_max
orb%_max
drb%_max
trb%_max
ast%_max
stl%_max
blk%_max
tov%_max
usg%_max
ortg_max
drtg_max
team
total
home
mp_opp
fg_opp
fga_opp
fg%_opp
3p_opp
3pa_opp
3p%_opp
ft_opp
fta_opp
ft%_opp
orb_opp
drb_opp
trb_opp
ast_opp
stl_opp
blk_opp
tov_opp
pf_opp
pts_opp
ts%_opp
efg%_opp
3par_opp
ftr_opp
orb%_opp
drb%_opp
trb%_opp
ast%_opp
stl%_opp
blk%_opp
tov%_opp
usg%_opp
ortg_opp
drtg_opp
fg_max_opp
fga_max_opp
fg%_max_opp
3p_max_opp
3pa_max_opp
3p%_max_opp
ft_max_opp
fta_max_opp
ft%_max_opp
orb_max_opp
drb_max_opp
trb_max_opp
ast_max_opp
stl_max_opp
blk_max_opp
tov_max_opp
pf_max_opp
pts_max_opp
gmsc_max_opp
+/-_max_opp
ts%_max_opp
efg%_max_opp
3par_max_opp
ftr_max_opp
orb%_m

## Difference Features
To reduce the complexity of the features and the potential for noise to reduce model performance, we can change the features to be the difference in stats between the two teams playing instead of inputting each team's performance.

For example, instead of having **"fg%_10"** and **"fg%_opp_10"**, we can just have **"fg%_diff"** where **fg%_diff = fg%_10 - fg%_opp_10**

In [102]:
def get_team_diff(df):
    #Select all _opp columns and remove invalid columns
    removed_columns = ["team","team_opp","home","home_opp","season","won"]
    selected_columns = df.columns[~df.columns.isin(removed_columns)]
    
    df["won_opp_10"] = df["won_10"] * -1
    
    #If a col has _opp, then it also has the same col for the other team
    selected_columns = [c for c in selected_columns if "_opp" in c]
    
    #Create new df
    df_diff_only = pd.DataFrame(index=df.index)
    
    #For each column, if there is another column with the exact string, without the _opp
    for col in selected_columns:
        other_col = col.replace("_opp", "")
        
        #Compute difference for each numeric column and Add to empty df
        if other_col in df.columns:
            df_diff_only[f"{other_col}_diff"] = df[other_col] - df[col]
            
    return df_diff_only

df_diff = get_team_diff(df)

C:\Users\henXc\AppData\Local\Temp/ipykernel_114684/1668524509.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  df_diff_only[f"{other_col}_diff"] = df[other_col] - df[col]


In [103]:
df_diff.head()

,mp_diff,fg_diff,fga_diff,fg%_diff,3p_diff,3pa_diff,3p%_diff,ft_diff,fta_diff,ft%_diff,...,stl%_max_10_diff,blk%_max_10_diff,tov%_max_10_diff,usg%_max_10_diff,ortg_max_10_diff,drtg_max_10_diff,total_10_diff,home_10_diff,prev_date_diff,days_rested_diff
2696,0.0,-2.0,2.0,-0.034,-2.0,-3.0,-0.040,-11.0,-12.0,-0.016,...,-0.5,7.30,6.25,-5.40,-47.0,3.5,0.0,0.0,1 days,-1.0
2681,0.0,-3.0,3.0,-0.050,-6.0,-2.0,-0.235,-7.0,-8.0,-0.050,...,2.3,2.45,6.95,13.70,-94.5,8.5,-6.5,1.0,1 days,-1.0
2694,0.0,7.0,2.0,0.076,6.0,-1.0,0.203,-14.0,-16.0,-0.111,...,0.6,5.45,10.80,-1.45,-43.5,1.5,-0.5,0.0,-1 days,1.0
2692,0.0,2.0,-4.0,0.043,-3.0,-9.0,-0.011,-7.0,-4.0,-0.156,...,-0.8,-3.55,-38.90,-4.35,-62.0,15.5,-19.0,0.0,0 days,0.0
2688,0.0,3.0,-3.0,0.050,6.0,2.0,0.235,7.0,8.0,0.050,...,3.8,9.90,9.35,-0.85,92.5,-19.0,18.5,-1.0,-1 days,1.0


In [59]:
for col in df_diff.columns:
    print(col)

mp_diff
mp_diff_10
fg_diff
fg_diff_10
fga_diff
fga_diff_10
fg%_diff
fg%_diff_10
3p_diff
3p_diff_10
3pa_diff
3pa_diff_10
3p%_diff
3p%_diff_10
ft_diff
ft_diff_10
fta_diff
fta_diff_10
ft%_diff
ft%_diff_10
orb_diff
orb_diff_10
drb_diff
drb_diff_10
trb_diff
trb_diff_10
ast_diff
ast_diff_10
stl_diff
stl_diff_10
blk_diff
blk_diff_10
tov_diff
tov_diff_10
pf_diff
pf_diff_10
pts_diff
pts_diff_10
ts%_diff
ts%_diff_10
efg%_diff
efg%_diff_10
3par_diff
3par_diff_10
ftr_diff
ftr_diff_10
orb%_diff
orb%_diff_10
drb%_diff
drb%_diff_10
trb%_diff
trb%_diff_10
ast%_diff
ast%_diff_10
stl%_diff
stl%_diff_10
blk%_diff
blk%_diff_10
tov%_diff
tov%_diff_10
usg%_diff
usg%_diff_10
ortg_diff
ortg_diff_10
drtg_diff
drtg_diff_10
fg_max_diff
fg_max_diff_10
fga_max_diff
fga_max_diff_10
fg%_max_diff
fg%_max_diff_10
3p_max_diff
3p_max_diff_10
3pa_max_diff
3pa_max_diff_10
3p%_max_diff
3p%_max_diff_10
ft_max_diff
ft_max_diff_10
fta_max_diff
fta_max_diff_10
ft%_max_diff
ft%_max_diff_10
orb_max_diff
orb_max_diff_10
drb_max_d

In [60]:
df = pd.concat([df[removed_columns], df_diff], axis=1)

In [61]:
df.head()

,team,team_opp,home,season,won,mp_diff,mp_diff_10,fg_diff,fg_diff_10,fga_diff,...,usg%_max_diff,usg%_max_diff_10,ortg_max_diff,ortg_max_diff_10,drtg_max_diff,drtg_max_diff_10,total_diff,total_diff_10,prev_date_diff,days_rested_diff
2696,IND,CHI,0,2017,False,0.0,0.0,-2.0,-2.0,2.0,...,3.9,3.9,-32.0,-32.0,18.0,18.0,-17,-17,1 days,-1.0
2681,NOP,SAS,0,2017,False,0.0,0.0,-3.0,-3.0,3.0,...,4.2,4.2,45.0,45.0,18.0,18.0,-19,-19,1 days,-1.0
2694,BOS,CHO,0,2017,True,0.0,0.0,7.0,7.0,2.0,...,-6.9,-6.9,57.0,57.0,-5.0,-5.0,6,6,-1 days,1.0
2692,ORL,CLE,0,2017,False,0.0,0.0,2.0,2.0,-4.0,...,-3.8,-3.8,-38.0,-38.0,8.0,8.0,-6,-6,0 days,0.0
2688,SAS,NOP,1,2017,True,0.0,0.0,3.0,3.0,-3.0,...,-4.2,-4.2,-45.0,-45.0,-18.0,-18.0,19,19,-1 days,1.0


### Balancing the dataset 

In [19]:
print(len(df[df['won'] == 1]))
print(len(df[df['won'] == 0]))

3734
3733


In [20]:
from sklearn.utils import resample

def balance_dataframe(df):
    wins = df[df['won'] == 1]
    losses = df[df['won'] == 0]
    
    if len(wins) == len(losses): return df
    if len(wins) > len(losses):
        wins_down = resample(wins,
                   replace=False,
                   n_samples=len(losses),
                   random_state=42)

        return pd.concat([losses, wins_down])
    
    losses_down = resample(losses,
               replace=False,
               n_samples=len(wins),
               random_state=42)
    return pd.concat([wins, losses_down])

df = balance_dataframe(df)

In [21]:
print(len(df[df['won'] == 1]))
print(len(df[df['won'] == 0]))

3733
3733


# Final Dataframe
The final features must only include past data and current information, that can both be inputted into the model.

For the ai model to predict which team is going to win, it needs the recent stats from each team playing.

### The primary key used for finding the most recent game played and pull statistics from that game
- season
- date
- team (home team)
- team_opp (away team)

### Information the model uses to make predictions
- rolling 10-game averages team (contains '_10')
- rolling 10-game averages team_opp (contains '_opp_10')
- rolling 5-game averages team (contains '_5')
- rolling 5-game averages team_opp (contains '_opp_5')
- team performance against team_opp in past 3 games
- team_opp performance against team in past 3 games
- days rested team
- days rested team_opp

### Label/Outcome/Loss/Class
- won   <-- used as label, test model performance with this feature

In [22]:
cols_10 = [col for col in df.columns if "_10" in col]
selected_columns = ["season","date","team","team_opp"] + cols_10 + ["days_rested","days_rested_opp"] + ["won"]

In [23]:
for column in selected_columns:
    print(column)

season
date
team
team_opp
mp_10
fg_10
fga_10
fg%_10
3p_10
3pa_10
3p%_10
ft_10
fta_10
ft%_10
orb_10
drb_10
trb_10
ast_10
stl_10
blk_10
tov_10
pf_10
pts_10
ts%_10
efg%_10
3par_10
ftr_10
orb%_10
drb%_10
trb%_10
ast%_10
stl%_10
blk%_10
tov%_10
usg%_10
ortg_10
drtg_10
fg_max_10
fga_max_10
fg%_max_10
3p_max_10
3pa_max_10
3p%_max_10
ft_max_10
fta_max_10
ft%_max_10
orb_max_10
drb_max_10
trb_max_10
ast_max_10
stl_max_10
blk_max_10
tov_max_10
pf_max_10
pts_max_10
gmsc_max_10
+/-_max_10
ts%_max_10
efg%_max_10
3par_max_10
ftr_max_10
orb%_max_10
drb%_max_10
trb%_max_10
ast%_max_10
stl%_max_10
blk%_max_10
tov%_max_10
usg%_max_10
ortg_max_10
drtg_max_10
total_10
home_10
mp_opp_10
fg_opp_10
fga_opp_10
fg%_opp_10
3p_opp_10
3pa_opp_10
3p%_opp_10
ft_opp_10
fta_opp_10
ft%_opp_10
orb_opp_10
drb_opp_10
trb_opp_10
ast_opp_10
stl_opp_10
blk_opp_10
tov_opp_10
pf_opp_10
pts_opp_10
ts%_opp_10
efg%_opp_10
3par_opp_10
ftr_opp_10
orb%_opp_10
drb%_opp_10
trb%_opp_10
ast%_opp_10
stl%_opp_10
blk%_opp_10
tov%_opp_10
us

In [24]:
df[selected_columns].head()

,season,date,team,team_opp,mp_10,fg_10,fga_10,fg%_10,3p_10,3pa_10,...,tov%_max_opp_10,usg%_max_opp_10,ortg_max_opp_10,drtg_max_opp_10,total_opp_10,home_opp_10,won_10,days_rested,days_rested_opp,won
2696,2017,2016-10-29,IND,CHI,252.5,42.0,95.5,0.4415,8.5,21.5,...,25.00,33.75,189.5,111.0,112.0,0.5,0.5,1.0,2.0,False
2681,2017,2016-10-29,NOP,SAS,240.0,42.5,96.0,0.4415,4.0,20.5,...,34.70,27.50,225.0,107.5,114.5,0.0,0.0,1.0,2.0,False
2692,2017,2016-10-29,ORL,CLE,240.0,33.5,92.0,0.3645,7.0,22.0,...,58.05,31.65,188.0,105.0,108.0,0.5,0.0,1.0,1.0,False
2709,2017,2016-10-30,DAL,HOU,252.5,40.0,92.0,0.4355,13.0,38.5,...,31.25,38.20,167.5,114.0,118.0,0.5,0.0,2.0,2.0,False
2702,2017,2016-10-30,LAL,OKC,240.0,38.5,86.5,0.4435,10.5,29.0,...,65.00,37.95,176.5,118.0,105.0,0.5,0.5,2.0,2.0,False


In [25]:
df = df[selected_columns]

In [26]:
df.to_pickle('df_nba_games.pkl')